# Reasoning Driving: `experiment.ipynb`

EGN 6216: Test A and Test B with the model.

Does giving the model the region's rule text help it read lane markings? Every labeled image is sent to the model twice:

| | What the model gets | What it answers |
|---|---|---|
| **Test A** | the photo with the target marking in a cyan box + the task | Output 1 (marking type, arrow type, position), Output 2 (6 actions), evidence |
| **Test B** | the same + the rule passages of the photo's region (MUTCD for the US, GB 5768.3 for China) | the same + the ID of the passage it relied on |

Everything else (task text, answer format, model, settings, image) is identical, so a difference between A and B comes from the rule text.

* Settings: `config/experiment.yaml`. Prompts: `prompts/v3/` (default), `prompts/v1/` and `prompts/v2/`. Code: `lane_reasoner/`.
* Results: `runs/<run_id>/` (settings, prompt hashes, raw answers, scores).
* `experiment.mock: true` (the default) makes up answers without calling the API, so everything can be tried for free. Set it to `false` (or use the switch on the demo page) for real calls.

API key: Section 1 asks for your key in a hidden box and saves it in `.env`.

## 0. Setup

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

REPO = Path.cwd()                      # this notebook sits in the repository folder
assert (REPO / "lane_reasoner").is_dir(), "Open this notebook from the repository folder."
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import importlib
import lane_reasoner
from lane_reasoner import config as C
for m in list(sys.modules):            # pick up code changes without restarting the kernel
    if m.startswith("lane_reasoner."):
        importlib.reload(sys.modules[m])

cfg = C.load(REPO)
from lane_reasoner.data import workspace_dirs
for w in workspace_dirs(cfg):
    print(("found   " if w.is_dir() else "missing ") + str(w))
print(f"\nlane_reasoner {lane_reasoner.__version__} · mode: {'MOCK (no API calls)' if cfg['mock'] else 'REAL API calls'}")

## 1. API key (once)

Run the cell, paste your key into the hidden box and press Enter (an empty entry keeps the saved key). The key is written to `.env` as `OPENAI_API_KEY=...`; to remove it, delete that line.

In [ ]:
import getpass

ENV = REPO / ".env"
NAME = cfg["api"].get("api_key_env", "OPENAI_API_KEY")
_k = getpass.getpass(f"{NAME} (hidden; Enter = keep the current one): ").strip()
if _k:
    lines = ENV.read_text(encoding="utf-8").splitlines() if ENV.exists() else []
    lines = [l for l in lines if not l.strip().startswith(NAME + "=")] + [f"{NAME}={_k}"]
    ENV.write_text("\n".join(lines) + "\n", encoding="utf-8")
    try:
        os.chmod(ENV, 0o600)           # owner-only, where the file system supports it
    except OSError:
        pass
    os.environ[NAME] = _k
del _k

try:
    ignored = subprocess.run(["git", "check-ignore", "-q", str(ENV)], cwd=REPO).returncode == 0
except FileNotFoundError:
    ignored = ".env" in (REPO / ".gitignore").read_text(encoding="utf-8").split()
pub = C.public(cfg)["api"]
print("Key:", "set" if pub["api_key_set"] else "not set")
print(".env ignored by Git:", "yes" if ignored else "no: add .env to .gitignore")

## 2. Settings

All settings are in `config/experiment.yaml`. The most important ones:

| Setting | Meaning |
|---|---|
| `experiment.mock` | `true`: made-up answers, no API call, no cost. `false`: real calls |
| `experiment.use_rules` | `no`: Test A (image only); `yes`: Test B (image + rule passages) |
| `experiment.region` | which region's images and rules are used: `US` or `CN` (any code under `regions`) |
| `experiment.prompt` | prompt folder under `prompts/` |
| `model.name`, `model.reasoning_effort`, `model.reasoning_summary`, `model.temperature`, `model.image_detail`, `model.max_output_tokens` | request settings, the same for Test A and Test B |
| `data.workspaces` | the data folders whose final labels are used (own video, BDD100K, CULane) |
| `regions.<code>` | the region's name in the prompt and its rules (passages file, code, title, language) |

In Python the settings are a dictionary: `cfg["mock"]`, `cfg["run"]["test"]` (`"A"` or `"B"`), `cfg["run"]["region"]`, `cfg["api"]["model"]`, `cfg["prompt"]["dir"]`. The cell below changes them for this session only (the demo page can do the same). Edit the file, or call `C.save(cfg)`, to keep them.

In [ ]:
cfg["mock"] = True          # True: made-up answers (free); False: real API calls
cfg["run"]["test"] = "A"
cfg["run"]["region"] = "US"
C.check(cfg)

from lane_reasoner.data import load_items
import pandas as pd

pub = C.public(cfg); pub["api"].pop("api_key_hint", None)
print(json.dumps({k: pub[k] for k in ("mock", "api", "run")}, indent=2))
counts = {r: len(load_items(cfg, r, "any", [])) for r in cfg["regions"]}
print("\nLabeled images per region:", counts)
items = load_items(cfg, cfg["run"]["region"], "any", [])
pd.DataFrame([{"image_id": i.image_id, "collection": i.collection, "split": i.split, "marking": i.marking,
               "visibility": i.label.get("visibility")} for i in items]).value_counts("marking").to_frame("images")

## 3. The prompt: what Test A and Test B send

One request = an **instructions** message (developer role) + a **user** message with one sentence and the boxed image, and a **JSON schema** that fixes the answer format (strict mode, so the answer always parses).

Test B differs only in three places: step 2 of the reasoning ("find the rule passage below" instead of "recall the rules"), the `cited_rule_id` line, and the block of rule passages at the end of the instructions.

In [ ]:
from lane_reasoner.prompts import prompt_parts

REGION = cfg["run"]["region"]
pa, pb = prompt_parts(cfg, "A", REGION), prompt_parts(cfg, "B", REGION)
print("=== Test A: instructions ===\n" + pa["instructions"])
print("\n=== user message (both tests; the image follows) ===\n" + pa["user_text"])
la, lb = set(pa["instructions"].splitlines()), pb["instructions"].splitlines()
print("\n=== only in Test B ===")
print("\n".join(l for l in lb if l not in la))
print("\nshared parts identical:", pa["hashes"]["shared_parts"] == pb["hashes"]["shared_parts"],
      f"· approx. prompt length A {len(pa['instructions'])//4} tokens, B {len(pb['instructions'])//4} tokens (+ image)")

## 4. Self-test (free, no API)

Checks the whole pipeline on synthetic images with a fake API server: labels, both prompts, the boxed image, scoring, runs, comparison, request format, retries, errors, cache, the demo server, and that the key is not written to any file.

In [ ]:
r = subprocess.run([sys.executable, "-m", "lane_reasoner", "selftest"], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-3000:])

## 5. Check the key and the model (1 tiny request)

One short text request with the same model and settings as the real runs. It costs a fraction of a cent and tells you at once if the key, the model name or a setting (for example `temperature` or `reasoning_effort`) is not accepted. The error message names the setting.

In [ ]:
from lane_reasoner.client import ApiError, ping
if cfg["mock"]:
    print("Mock mode: nothing sent. Set cfg['mock'] = False in Section 2 to check the key and the model.")
else:
    try:
        res = ping(cfg)
        print(f"OK · model {res['model']} answered {res['text']!r} in {res['latency_s']} s · tokens {res['usage'].get('input_tokens')}/{res['usage'].get('output_tokens')}")
    except ApiError as e:
        print("Not OK:", e)

## 6. Smoke test: a few real images

Asks the model about the first `N` labeled images (fixed random order), Test A and Test B, prints the reasoning of each answer step by step, and shows the answers next to the labels. Real calls only when `cfg["mock"]` is `False` (Section 2). Answers are cached in `runs/cache/`, so running this again costs nothing.

In [ ]:
from lane_reasoner.runner import ask, cost_of
from lane_reasoner.trace import passage_map, trace_text

N = 3
USE_API = not cfg["mock"]   # follows the switch in Section 2
rows, spent = [], 0.0
for it in load_items(cfg, REGION, "any", [])[:N]:
    for test in ("A", "B"):
        rec = ask(cfg, it, test, 0, mock=not USE_API)
        print(trace_text(rec, passage_map(cfg, it.region)) + "\n")
        a, s = rec.get("answer") or {}, rec.get("score") or {}
        if not rec.get("cached"):
            spent += cost_of(cfg, (rec.get("response") or {}).get("usage") or {})
        rows.append({"image": it.image_id[-14:], "test": test, "label": it.marking,
                     "answer": (a.get("marking") or {}).get("marking_type", rec.get("error", "")[:60]),
                     "out1": s.get("output1"), "out2": s.get("output2"), "evidence": s.get("evidence_group"),
                     "citation": s.get("citation"), "tokens": ((rec.get("response") or {}).get("usage") or {}).get("input_tokens"),
                     "cached": rec.get("cached")})
print(("REAL answers" if USE_API else "MOCK answers") + f" · cost of this cell ≈ ${spent:.4f}")
pd.DataFrame(rows)

## 7. Demo page (for presenting)

Opens a page in the browser: pick an image, see the exact prompt of Test A or B, ask the model, see the answer next to the label, run a batch and compare two runs. It runs in the background on your computer only (127.0.0.1). The switches on the page change the settings of this notebook session too.

In [ ]:
from lane_reasoner import demo
url = demo.start(cfg)        # demo.stop() ends it
print("Demo:", url)

## 8. Batch runs: Test A and Test B on all labeled images

Each run writes `runs/<run_id>/manifest.json` (settings, prompt hashes, label hashes, Git commit, image list), `results.jsonl` (one line per request: settings, raw answer, parsed answer, score), `scores.json` and `summary.md`.

* `LIMIT = 0` uses every labeled image of the region. `REPEATS = 3` asks three times and takes the majority; the flip rate shows how stable the answers are.
* Check the cost of the smoke test first: a run costs about (images × repeats × 2 tests) times the cost of one request.

In [ ]:
from lane_reasoner.runner import run_batch, compare_runs, load_run

USE_API = not cfg["mock"]   # follows the switch in Section 2
LIMIT, REPEATS = 0, 1       # 0 = all labeled images of the region
from lane_reasoner.runner import run_pair
ids = run_pair(cfg, REGION, LIMIT, REPEATS, "any", mock=not USE_API)   # Test A and Test B at the same time
run_a, run_b = ids["A"], ids["B"]

In [ ]:
c = compare_runs(cfg, run_a, run_b)
print(f"A = {run_a}\nB = {run_b}\nsame settings: {c['same_settings']} · same shared prompt parts: {c['same_shared_prompt']}\n")
display(pd.DataFrame({k: c[k] for k in ("output1", "output2")}).T.rename(columns={
    "n": "paired images", "acc_A": "Test A", "acc_B": "Test B", "only_B_right": "only B right (b)",
    "only_A_right": "only A right (c)", "p_value": "exact McNemar p"}))
sa, sb = (load_run(cfg, r)["scores"]["summary"] for r in (run_a, run_b))
display(pd.DataFrame({"Test A": sa["evidence_groups"], "Test B": sb["evidence_groups"]}).fillna(0).astype(int))
print("Citations in Test B:", sb.get("citations"))
print("Unsafe errors (label prohibited, answer allowed): A", sa["unsafe_errors"], "· B", sb["unsafe_errors"])

## 9. Adding a region (for example China, or another country)

1. Put the region's passages in a file in `rules/`, for example `rules/jp_rules.yaml` (each passage: `rule_id`, `region: JP`, `source`, `section`, `page`, `marking_type`, `text_en`, original text), and add its rows to `config/action_rules.json`.
2. Add an entry under `regions` in `config/experiment.yaml`:
   ```yaml
   JP:
     name: Japan
     rules: {file: rules/jp_rules.yaml, code: JP, title: "...", language: en}
   ```
3. Label its images with `video_tool` (a workspace whose labels have `region` = `JP`), add the folder to `data.workspaces` and set `experiment.region: JP`.

The prompt files do not change: the region name and the passages are filled in from these files. China (`CN`) is already set up; it only needs labeled images.